In [1]:
# Cell 1: Setup & Installations
import sys

if "google.colab" in sys.modules or True:
    print("Installing required packages...")
    %pip install -q langchain>=0.1.0 langchain-openai>=0.0.5 langchain-community>=0.0.20 langchain-text-splitters>=0.2.0 chromadb>=0.4.0 tiktoken>=0.5.0 python-dotenv>=1.0.0

print("Packages ready")

Installing required packages...
Note: you may need to restart the kernel to use updated packages.
Packages ready


c:\Users\viraj\Zuu\Real_State_Agent\mp_02\Scripts\python.exe: No module named pip


In [2]:
# Cell 2: Imports & Environment Setup
import os
import sys
import json
import random
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

groq_key = os.getenv("GROQ_API_KEY")
openrouter_key = os.getenv("OPENROUTER_API_KEY")

if not groq_key and not openrouter_key:
    raise EnvironmentError(
        "No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or GROQ_API_KEY to .env"
    )

# Load configuration
from context_engineering.config import (
    CRAWL_OUT_DIR, VECTOR_DIR, EMBEDDING_MODEL, PROVIDER
)

random.seed(42)

provider = "groq"
print("Environment loaded")
print(f"Provider: {provider}")
print(f"Project root: {project_root}")

Environment loaded
Provider: groq
Project root: c:\Users\viraj\Zuu\Real_State_Agent


In [3]:
# Cell 3: Import Chunking Services
from context_engineering.application.ingest_document_service.chunkers import (
    semantic_chunk,
    fixed_chunk,
    sliding_chunk,
    parent_child_chunk,
    late_chunk_index,
    late_chunk_split,
    ChunkingService,
    count_tokens,
 )

print("Chunking services loaded from service layer")
print(" Location: context_engineering.application.ingest_document_service.chunkers")
print("\n Available strategies:")
print("   1. semantic_chunk   - Split by heading structure")
print("   2. fixed_chunk      - Uniform token chunks with overlap")
print("   3. sliding_chunk    - Overlapping windows for better recall")
print("   4. parent_child     - Parent-child two-tier chunking")
print("   5. late_chunk_index - Base passages for late splitting")
print("   6. late_chunk_split - Retrieval-time split near matches")

Chunking services loaded from service layer
 Location: context_engineering.application.ingest_document_service.chunkers

 Available strategies:
   1. semantic_chunk   - Split by heading structure
   2. fixed_chunk      - Uniform token chunks with overlap
   3. sliding_chunk    - Overlapping windows for better recall
   4. parent_child     - Parent-child two-tier chunking
   5. late_chunk_index - Base passages for late splitting
   6. late_chunk_split - Retrieval-time split near matches


### Load Data

In [4]:
# Cell 4: Load Corpus
jsonl_path = CRAWL_OUT_DIR / "primelands_docs.jsonl"

if not jsonl_path.exists():
    raise FileNotFoundError(f" Corpus not found. Run run_primelands_crawl.py first.")

with open(jsonl_path, 'r', encoding='utf-8') as f:
    documents = [json.loads(line) for line in f]

print(f" Loaded {len(documents)} documents")
print(f"Total content size: {sum(len(d['content']) for d in documents):,} chars")

 Loaded 270 documents
Total content size: 924,095 chars


### Apply Chunking

In [5]:
# Cell: Cleanup Vector Store (prevents corruption)
import shutil

# Remove existing vector store to prevent corruption issues
if VECTOR_DIR.exists():
    print(f"  Removing existing vector store: {VECTOR_DIR}")
    shutil.rmtree(VECTOR_DIR)
    print("    Cleaned up")

# Create fresh directory
VECTOR_DIR.mkdir(parents=True, exist_ok=True)
print(f" Fresh vector directory ready: {VECTOR_DIR}")


  Removing existing vector store: c:\Users\viraj\Zuu\Real_State_Agent\data\vectorstore
    Cleaned up
 Fresh vector directory ready: c:\Users\viraj\Zuu\Real_State_Agent\data\vectorstore


In [6]:
# Semantic Chunking (using service!)
print(" Running semantic chunking...")
semantic_chunks = semantic_chunk(documents)

semantic_path = CRAWL_OUT_DIR / "chunks_semantic.jsonl"
with open(semantic_path, "w", encoding="utf-8") as f:
    for chunk in semantic_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f" Semantic chunking complete: {len(semantic_chunks)} chunks")
print(f" Saved to: {semantic_path}")

 Running semantic chunking...
 Semantic chunking complete: 338 chunks
 Saved to: c:\Users\viraj\Zuu\Real_State_Agent\data\chunks_semantic.jsonl


In [7]:
# Fixed Chunking (using service!)
print(" Running fixed chunking...")
fixed_chunks = fixed_chunk(documents)

fixed_path = CRAWL_OUT_DIR / "chunks_fixed.jsonl"
with open(fixed_path, "w", encoding="utf-8") as f:
    for chunk in fixed_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f" Fixed chunking complete: {len(fixed_chunks)} chunks")
print(f" Saved to: {fixed_path}")

 Running fixed chunking...
 Fixed chunking complete: 464 chunks
 Saved to: c:\Users\viraj\Zuu\Real_State_Agent\data\chunks_fixed.jsonl


In [8]:
# Sliding Chunking (using service!)
print(" Running sliding chunking...")
sliding_chunks = sliding_chunk(documents)

sliding_path = CRAWL_OUT_DIR / "chunks_sliding.jsonl"
with open(sliding_path, "w", encoding="utf-8") as f:
    for chunk in sliding_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f" Sliding chunking complete: {len(sliding_chunks)} chunks")
print(f" Saved to: {sliding_path}")

 Running sliding chunking...
 Sliding chunking complete: 1160 chunks
 Saved to: c:\Users\viraj\Zuu\Real_State_Agent\data\chunks_sliding.jsonl


In [9]:
# Parent-Child Chunking (using service!)
print(" Running parent-child chunking...")
child_chunks, parent_chunks = parent_child_chunk(documents)

children_path = CRAWL_OUT_DIR / "chunks_parent_child_children.jsonl"
parents_path = CRAWL_OUT_DIR / "chunks_parent_child_parents.jsonl"
with open(children_path, "w", encoding="utf-8") as f:
    for chunk in child_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")
with open(parents_path, "w", encoding="utf-8") as f:
    for chunk in parent_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f" Parent-child chunking complete: {len(child_chunks)} children, {len(parent_chunks)} parents")
print(f" Saved children to: {children_path}")
print(f" Saved parents to: {parents_path}")

 Running parent-child chunking...
 Parent-child chunking complete: 1401 children, 324 parents
 Saved children to: c:\Users\viraj\Zuu\Real_State_Agent\data\chunks_parent_child_children.jsonl
 Saved parents to: c:\Users\viraj\Zuu\Real_State_Agent\data\chunks_parent_child_parents.jsonl


In [10]:
# Late Chunk Base Indexing (using service!)
print("Running late chunk base indexing...")
late_base_chunks = late_chunk_index(documents)

late_base_path = CRAWL_OUT_DIR / "chunks_late_base.jsonl"
with open(late_base_path, "w", encoding="utf-8") as f:
    for chunk in late_base_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Late chunk base indexing complete: {len(late_base_chunks)} chunks")
print(f" Saved to: {late_base_path}")

Running late chunk base indexing...
Late chunk base indexing complete: 341 chunks
 Saved to: c:\Users\viraj\Zuu\Real_State_Agent\data\chunks_late_base.jsonl


### Check Samples

In [11]:
# Spot-Check Samples
print("🔍 Spot-Check: 2 samples from each strategy\n")

def print_sample(chunk, strategy_name):
    print(f"**{strategy_name}** chunk:")
    print(f"  URL: {chunk['url']}")
    print(f"  Strategy: {chunk['strategy']}")
    print(f"  Text length: {len(chunk['text'])} chars")
    print(f"  Preview: {chunk['text'][:100]}...")
    print()

print("=" * 60)
print("SEMANTIC SAMPLES")
print("=" * 60)
for chunk in random.sample(semantic_chunks, min(2, len(semantic_chunks))):
    print_sample(chunk, "Semantic")

print("=" * 60)
print("FIXED-WINDOW SAMPLES")
print("=" * 60)
for chunk in random.sample(fixed_chunks, min(2, len(fixed_chunks))):
    print_sample(chunk, "Fixed")

print("=" * 60)
print("SLIDING-WINDOW SAMPLES")
print("=" * 60)
for chunk in random.sample(sliding_chunks, min(2, len(sliding_chunks))):
    print_sample(chunk, "Sliding")

🔍 Spot-Check: 2 samples from each strategy

SEMANTIC SAMPLES
**Semantic** chunk:
  URL: https://www.primelands.lk/land/ZEAL-MALABE/en
  Strategy: semantic
  Text length: 2573 chars
  Preview: # 1,600,000 LKR

PER PERCH UPWARDS

![location_dark.png](https://www.primelands.lk/public/assets/ima...

**Semantic** chunk:
  URL: https://www.primelands.lk/land/ATH-PAURA-KURUNEGALA/en
  Strategy: semantic
  Text length: 3118 chars
  Preview: # 147,500 LKR

PER PERCH UPWARDS

![location_dark.png](https://www.primelands.lk/public/assets/image...

FIXED-WINDOW SAMPLES
**Fixed** chunk:
  URL: https://www.primelands.lk/apartment/YOLO-KIRIBATHGODA/en
  Strategy: fixed
  Text length: 394 chars
  Preview: ![250213080205Road_Map_-_YOLO.jpg](https://plcms.primelands.lk/images/250213080205Road_Map_-_YOLO.jp...

**Fixed** chunk:
  URL: https://www.primelands.lk/land/MAGNETISM-WELIGAMA/en
  Strategy: fixed
  Text length: 2675 chars
  Preview: ![location_dark.png](https://www.primelands.lk/public/assets/imag

### Chunking Comparison

In [12]:
# Comparison Table: Chunk Count, Avg Size, Index Size, Retrieval Time
import time

strategy_chunks = {
    "semantic": semantic_chunks,
    "fixed": fixed_chunks,
    "sliding": sliding_chunks,
    "parent_child_children": child_chunks,
    "late_base": late_base_chunks,
}

def avg_tokens(chunks):
    if not chunks:
        return 0
    if "token_count" in chunks[0]:
        return sum(c.get("token_count", 0) for c in chunks) / len(chunks)
    return sum(count_tokens(c["text"]) for c in chunks) / len(chunks)

def estimate_index_size_mb(count, dim):
    if not count or not dim:
        return 0
    return (count * dim * 4) / (1024 * 1024)

def try_get_dim_from_qdrant(name):
    try:
        info = client.get_collection(name)
        return info.config.params.vectors.size
    except Exception:
        return None

def try_retrieval_time_ms(name):
    if "client" not in globals() or "embed_texts" not in globals():
        return None
    try:
        query_vec = embed_texts(["example query"])[0]
        durations = []
        for _ in range(3):
            start = time.perf_counter()
            client.search(collection_name=name, query_vector=query_vec, limit=3)
            durations.append((time.perf_counter() - start) * 1000)
        return sum(durations) / len(durations)
    except Exception:
        return None

rows = []
for name, chunks in strategy_chunks.items():
    count = len(chunks)
    avg_size = avg_tokens(chunks)
    dim = None
    if "summary" in globals() and name in summary:
        dim = summary[name].get("dim")
    if not dim:
        dim = try_get_dim_from_qdrant(name)
    index_mb = estimate_index_size_mb(count, dim)
    rt_ms = try_retrieval_time_ms(name)
    rows.append({
        "strategy": name,
        "chunk_count": count,
        "avg_tokens": round(avg_size, 1),
        "index_size_mb": round(index_mb, 2),
        "retrieval_ms": None if rt_ms is None else round(rt_ms, 2),
    })

# Print as markdown table
header = "| Strategy | Chunk Count | Avg Size (tokens) | Index Size (MB) | Retrieval Time (ms) |"
sep = "|---|---:|---:|---:|---:|"
print(header)
print(sep)
for r in rows:
    rt = "n/a" if r["retrieval_ms"] is None else r["retrieval_ms"]
    print(f"| {r['strategy']} | {r['chunk_count']} | {r['avg_tokens']} | {r['index_size_mb']} | {rt} |")

| Strategy | Chunk Count | Avg Size (tokens) | Index Size (MB) | Retrieval Time (ms) |
|---|---:|---:|---:|---:|
| semantic | 338 | 778.1 | 0 | n/a |
| fixed | 464 | 605.7 | 0 | n/a |
| sliding | 1160 | 391.5 | 0 | n/a |
| parent_child_children | 1401 | 229.8 | 0 | n/a |
| late_base | 341 | 788.1 | 0 | n/a |


### Qdrant Indexing

In [13]:
# Qdrant Indexing (5 collections) - OpenRouter Embeddings (OpenAI SDK)
import os
import json
import time
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance, PointStruct
from openai import OpenAI

openrouter_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_key:
    raise EnvironmentError("OPENROUTER_API_KEY not found in .env")

client_openai = OpenAI(
    api_key=openrouter_key,
    base_url="https://openrouter.ai/api/v1"
 )

# Close any previous local client to release the lock
if "client" in globals():
    try:
        client.close()
    except Exception:
        pass

base_qdrant_path = VECTOR_DIR / "qdrant"
try:
    qdrant_path = base_qdrant_path
    client = QdrantClient(path=str(qdrant_path))
except RuntimeError:
    # Fallback to a unique path if the folder is locked by another instance
    qdrant_path = VECTOR_DIR / f"qdrant_{int(time.time())}"
    client = QdrantClient(path=str(qdrant_path))
    print(f"⚠️  Using alternate Qdrant path: {qdrant_path}")

def load_chunks_from_jsonl(path):
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                chunks.append(json.loads(line))
    return chunks

def ensure_chunks(obj):
    if isinstance(obj, (str, Path)):
        return load_chunks_from_jsonl(obj)
    return obj

def embed_texts(texts, batch_size=128):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client_openai.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch
        )
        if not resp.data:
            raise ValueError("No embedding data received")
        vectors.extend([item.embedding for item in resp.data])
    return vectors

def upsert_collection(name, chunks):
    chunks = ensure_chunks(chunks)
    if not chunks:
        print(f"Skipping {name}: no chunks")
        return 0, 0
    texts = [c["text"] for c in chunks if c.get("text")]
    vectors = embed_texts(texts)
    vector_size = len(vectors[0])
    client.recreate_collection(
        collection_name=name,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )
    points = []
    for idx, (chunk, vec) in enumerate(zip(chunks, vectors)):
        payload = {
            "url": chunk.get("url"),
            "title": chunk.get("title"),
            "strategy": chunk.get("strategy"),
            "chunk_index": chunk.get("chunk_index"),
        }
        if "heading" in chunk:
            payload["heading"] = chunk.get("heading")
        if "parent_id" in chunk:
            payload["parent_id"] = chunk.get("parent_id")
        points.append(PointStruct(id=idx, vector=vec, payload=payload))
    client.upsert(collection_name=name, points=points)
    return len(points), vector_size

collections = {
    "semantic": semantic_chunks,
    "fixed": fixed_chunks,
    "sliding": sliding_chunks,
    "parent_child_children": child_chunks,
    "late_base": late_base_chunks,
}

summary = {}
for name, chunks in collections.items():
    count, dim = upsert_collection(name, chunks)
    summary[name] = {"points": count, "dim": dim}

print("Qdrant indexing complete")
for name, info in summary.items():
    print(f"- {name}: {info['points']} points, dim {info['dim']}")

C:\Users\viraj\AppData\Local\Temp\ipykernel_28360\3857963804.py:70: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


ValueError: No embedding data received